# SCP Step 5 - Simulate

Run a prepared tune directory through the Step 5 backend. This notebook is intentionally thin: model setup, input generation, simulation, and saving are handled by `modules.simulation.SimulationSession`.


In [ ]:
# Local notebook bootstrap: autoreload modules and locate the SCP repo
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

try:
    from modules.notebook_helpers import ensure_scp_repo_on_syspath
except ModuleNotFoundError:
    start = Path.cwd().resolve()
    injected = False
    for cand in (start, *start.parents):
        if (cand / "modules").is_dir() and (cand / "run_pipeline.py").is_file():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            injected = True
            break
    if not injected:
        raise
    from modules.notebook_helpers import ensure_scp_repo_on_syspath

repo_root = ensure_scp_repo_on_syspath()
os.environ["SCP_ROOT"] = str(repo_root)
print("SCP repo:", repo_root)


In [ ]:
# User settings: choose the model and optional run overrides
cell_name = "SST"      # bundled examples: "PV" or "SST"
tune_name = "seg_tuned"

tune_dir = repo_root / "cells" / cell_name / "tunes" / tune_name
if not tune_dir.is_dir():
    raise FileNotFoundError(f"Tune directory not found: {tune_dir}")

# Keep these as None/False to use cell_configs/sim_config.json directly.
run_mode = None              # None, "single", or "multi"
n_trials = None              # e.g. 1 for a quick smoke run
seed = None                  # e.g. 1234 for reproducibility
run_iclamp = False           # True for current injection instead of synaptic inputs
force_save = False           # True to save even if sim_config disables saving
output_stem = None           # e.g. "notebook_test"; None uses sim_config/defaults
output_dir = None            # None -> tune_dir/output_data
preview_synapses = False     # True for a lightweight synapse-placement preview

options = SimulationOptions(
    mode=run_mode,
    n_trials=n_trials,
    seed=seed,
    iclamp=run_iclamp,
    force_save=force_save,
    output_stem=output_stem,
    output_dir=output_dir,
)

print("Tune dir:", tune_dir)


In [ ]:
# Prepare the simulation session
# Colab bootstrap defines ensure_modfiles(); local notebooks can ignore it.
try:
    ensure_modfiles(tune_dir, compile_modfiles=IN_COLAB)
except NameError:
    pass

session = SimulationSession.from_tune(tune_dir, options=options)
session.prepare()

print("Session summary:")
for key, value in session.summary().items():
    print(f"  {key}: {value}")

if session.groups_cfg:
    print("Synapse groups:", ", ".join(session.groups_cfg.keys()))
else:
    print("Synapse groups: none (IClamp mode or no active groups)")


In [ ]:
# Optional: preview synapse placement without running the simulation
if preview_synapses and not session.iclamp_enabled:
    syn_state = session.preview_synapses(trial_idx=0)
    print("Synapse preview complete.")
    for group_name, records in syn_state.get("records", {}).items():
        print(f"  {group_name}: {len(records)} synapses")
elif preview_synapses:
    print("Skipping synapse preview because IClamp mode is enabled.")
else:
    print("Synapse preview disabled.")


In [ ]:
# Run the simulation and save according to sim_config/options
results = session.run()
saved_path = session.save()

if saved_path is None:
    print("Results not saved. Enable saving in sim_config or set force_save=True.")
else:
    print("Results saved to:", saved_path)


In [ ]:
# Minimal diagnostics: spike counts and Vm trace
def _spike_counts(spikes):
    if spikes is None:
        return []
    if isinstance(spikes, list):
        return [len(np.asarray(trial)) for trial in spikes]
    return [len(np.asarray(spikes))]

mode = results.get("mode")
spike_counts = _spike_counts(results.get("spikes"))
if spike_counts:
    print("Mode:", mode)
    print("Spike counts:", spike_counts)
    print("Mean spikes/trial:", float(np.mean(spike_counts)))
else:
    print("Mode:", mode)
    print("No spike vector found; this is expected for some IClamp runs.")

traces = results.get("traces", {}) or {}
T = traces.get("T")
V = traces.get("V")

if T is not None and V is not None:
    plt.figure(figsize=(8, 4))
    if isinstance(V, list):
        for idx, trace in enumerate(V[:3]):
            plt.plot(T, trace, lw=1.0, label=f"trial {idx}")
        if len(V) > 1:
            plt.legend()
    else:
        plt.plot(T, V, lw=1.2)
    plt.xlabel("Time (ms)")
    plt.ylabel("Vm (mV)")
    plt.title(f"{cell_name} / {tune_name} ({mode})")
    plt.tight_layout()
else:
    print("No Vm trace stored. Increase n_traces_to_save or cell_recording settings if needed.")


## Optional Utilities

Use this only when you want to inspect a previous saved run from disk. More detailed analysis remains in `6_analysis.ipynb`.


In [ ]:
# Optional: load a previous saved run
load_previous = False
previous_run = tune_dir / "output_data" / "<run_name>"  # folder or run_manifest.json

if load_previous:
    results = run_sim.load_results(previous_run)
    print("Loaded:", previous_run)
    print("Mode:", results.get("mode"))
